# Fase 6 — BERTopic → Taksonomi Aspek

**Input :** `sentence_bank.parquet` (Fase 5) — dipakai kalimat **Tier A/B, ≤ 2024-12-31** (anti-kebocoran)
**Output:**
- `data/processed/bertopic_model/` — model tersimpan
- `data/processed/topic_info.csv` — daftar topik + kata kunci
- `data/processed/topic_to_aspect.csv` — **TEMPLATE (diisi manual)** topik → aspek A1-A8/S1-S3
- `data/processed/sentences_topic.parquet` — tiap kalimat training + topik

## ⚡ KOMPUTASI & WAKTU
- Pertama kali: **download model ~1-2 GB** (`multilingual-e5-base`). Butuh internet.
- GPU disarankan (otomatis dipakai bila torch mendeteksi CUDA).
- Waktu: ~10 menit (GPU) / ~30-50 menit (CPU).

## Install (sekali)
```bash
# (opsional GPU) install torch CUDA dulu: lihat requirements-topic.txt
pip install -r requirements-topic.txt
```

In [ ]:
import os
from pathlib import Path
import pandas as pd
import yaml

if Path('src').exists():
    ROOT = Path('.').resolve()
elif Path('../src').exists():
    os.chdir('..'); ROOT = Path('.').resolve()
import sys; sys.path.insert(0, str(ROOT))
from src.topics import (
    TopicConfig, build_topic_model, embed_texts, detect_device,
    select_training_corpus, topic_diversity, coherence_cv,
    make_topic_aspect_template, ASPECTS,
)

cfg_yaml = yaml.safe_load(open(ROOT / 'config/config.yaml'))
INTERIM = ROOT / 'data/interim'
PROCESSED = ROOT / 'data/processed'; PROCESSED.mkdir(parents=True, exist_ok=True)

device = detect_device()
print('DEVICE:', device, '(cuda = pakai GPU)')

In [ ]:
# 1. Pilih korpus training: Tier A/B, <= 2024-12-31 (anti-kebocoran)
sb = pd.read_parquet(INTERIM / 'sentence_bank.parquet')
cutoff = cfg_yaml['date_range']['bertopic_train_end']   # 2024-12-31
corpus = select_training_corpus(sb, cutoff=cutoff, tiers=('A', 'B'))
print('korpus training (kalimat unik):', len(corpus))
print('rentang tanggal:', pd.to_datetime(corpus.published_at).min(), '->', pd.to_datetime(corpus.published_at).max())
texts = corpus['text'].tolist()

In [ ]:
# 2. Bangun model + embedding (ini yang BERAT; download model pertama kali)
STOPWORDS_ID = None  # bisa isi daftar stopword Indonesia bila mau
tcfg = TopicConfig(min_cluster_size=40, min_samples=10, seed=cfg_yaml['seed'], stopwords=STOPWORDS_ID)

topic_model, embedder = build_topic_model(tcfg, device=device)
embeddings = embed_texts(embedder, texts, tcfg)
print('embedding shape:', embeddings.shape)

In [ ]:
# 3. Fit BERTopic
topics, _ = topic_model.fit_transform(texts, embeddings)
info = topic_model.get_topic_info()
n_topics = (info.Topic != -1).sum()
outlier_ratio = (pd.Series(topics) == -1).mean()
print(f'jumlah topik: {n_topics} | outlier ratio: {outlier_ratio:.2%}')
info.head(25)

In [ ]:
# 4. Metrik kualitas topik
div = topic_diversity(topic_model, topk=10)
coh = coherence_cv(topic_model, texts, topk=10)
print(f'topic diversity : {div:.3f}  (target tinggi, mendekati 1)')
print(f'coherence c_v   : {coh}' if coh is not None else 'coherence: (gensim tak terpasang)')
print(f'jumlah topik    : {n_topics}  (target 20-40)')
print(f'outlier ratio   : {outlier_ratio:.2%}')

In [ ]:
# 5. Simpan model, topic_info, kalimat+topik, dan TEMPLATE pemetaan
topic_model.save(str(PROCESSED / 'bertopic_model'), serialization='safetensors', save_ctfidf=True)
info.to_csv(PROCESSED / 'topic_info.csv', index=False)

corpus = corpus.assign(topic=topics)
corpus.to_parquet(PROCESSED / 'sentences_topic.parquet', index=False)

template = make_topic_aspect_template(topic_model)
template.to_csv(PROCESSED / 'topic_to_aspect.csv', index=False)
print('tersimpan di data/processed/:')
print('  bertopic_model/, topic_info.csv, sentences_topic.parquet, topic_to_aspect.csv')
print('\nAspek target untuk pemetaan manual:')
for k, v in ASPECTS.items(): print(f'  {k}: {v}')

## 6. LANGKAH MANUAL — pemetaan topik → aspek

Buka `data/processed/topic_to_aspect.csv`. Untuk tiap baris topik, lihat kolom `keywords`,
lalu isi:
- `aspect_annotator_1` — kode aspek (A1-A8/S1-S3) menurut kamu
- `aspect_annotator_2` — kode aspek menurut partnermu (independen)
- `aspect_final` — hasil rekonsiliasi berdua

**Syarat selesai (spec):** ≥ 85% kalimat terpetakan ke aspek ≠ S3.

Setelah terisi, simpan CSV-nya. File ini dipakai di Fase 8 (klasifier aspek).

In [ ]:
# Lihat kata kunci tiap topik untuk membantu pemetaan manual
for tid in sorted(set(topics)):
    if tid == -1:
        continue
    kw = ', '.join(w for w, _ in topic_model.get_topic(tid)[:10])
    cnt = (pd.Series(topics) == tid).sum()
    print(f'Topik {tid:3d} (n={cnt:4d}): {kw}')